# 📜 نظام تحليل العقود باستخدام Knowledge Graphs
## المراحل 1-4 الكاملة

---

### 📋 جدول المحتويات:

| المرحلة | الوصف |
|---------|-------|
| **Phase 1** | Knowledge Graphs الأساسية |
| **Phase 2** | التحليل المنطقي (Reasoning) + نسب لكل قانون |
| **Phase 3** | تحميل PDF (ملف/مجلد) + واجهة Gradio |
| **Phase 4** | تدريب النماذج (Train/Val/Test) + التقييم |

---

### ✨ المميزات:
- ✅ Knowledge Graphs للقوانين والعقود
- ✅ تحميل PDF (ملف واحد أو مجلد)
- ✅ نسب توافق منفصلة لكل قانون
- ✅ تحليل منطقي (Reasoning) مفصل
- ✅ تدريب نماذج ML مع Train/Val/Test
- ✅ واجهة Gradio تفاعلية
- ✅ يعمل محلياً بدون APIs مدفوعة

---

# 🔧 Phase 1: الأساسيات

---

## 1.1 تثبيت المكتبات

In [ ]:
# تثبيت المكتبات
!pip install -q networkx sentence-transformers faiss-cpu
!pip install -q PyPDF2 pdfplumber python-docx
!pip install -q pandas numpy scikit-learn
!pip install -q plotly matplotlib seaborn
!pip install -q reportlab gradio pyvis

## 1.2 استيراد المكتبات

In [ ]:
# === الاستيرادات ===
import os
import re
import json
import hashlib
import pickle
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Union
from dataclasses import dataclass, field, asdict
from enum import Enum

# معالجة البيانات
import numpy as np
import pandas as pd
import networkx as nx

# معالجة المستندات
import PyPDF2
import pdfplumber
from docx import Document as DocxDocument

# ML والتضمينات
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler

# العرض
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from pyvis.network import Network
from IPython.display import HTML, display

# تقارير PDF
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer

print("✅ تم استيراد جميع المكتبات")

## 1.3 التعريفات الأساسية (Enums & Data Classes)

In [ ]:
# === مستويات الخطر ===
class RiskLevel(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"
    
    @property
    def color(self):
        return {"low": "#28a745", "medium": "#ffc107", "high": "#fd7e14", "critical": "#dc3545"}[self.value]
    
    @property
    def emoji(self):
        return {"low": "🟢", "medium": "🟡", "high": "🟠", "critical": "🔴"}[self.value]
    
    @property
    def arabic(self):
        return {"low": "منخفض", "medium": "متوسط", "high": "عالي", "critical": "حرج"}[self.value]


class ComplianceStatus(Enum):
    COMPLIANT = "compliant"
    PARTIAL = "partial"
    NON_COMPLIANT = "non_compliant"
    NOT_APPLICABLE = "not_applicable"
    
    @property
    def icon(self):
        return {"compliant": "✅", "partial": "⚠️", "non_compliant": "❌", "not_applicable": "➖"}[self.value]


# === القوانين المدعومة ===
LAWS = {
    "PDPL": {"ar": "نظام حماية البيانات الشخصية", "en": "Personal Data Protection Law", "weight": 1.2, "color": "#3498db"},
    "ECC": {"ar": "نظام التجارة الإلكترونية", "en": "E-Commerce Law", "weight": 1.0, "color": "#9b59b6"},
    "LABOR": {"ar": "نظام العمل", "en": "Labor Law", "weight": 1.1, "color": "#e74c3c"},
    "CYBER": {"ar": "نظام مكافحة الجرائم المعلوماتية", "en": "Anti-Cyber Crime Law", "weight": 1.3, "color": "#2ecc71"},
    "COMMERCIAL": {"ar": "نظام الشركات", "en": "Companies Law", "weight": 1.0, "color": "#f39c12"},
    "ANTI_FRAUD": {"ar": "نظام مكافحة الاحتيال", "en": "Anti-Fraud Law", "weight": 1.2, "color": "#1abc9c"},
}

In [ ]:
# === هياكل البيانات ===

@dataclass
class LawArticle:
    """مادة قانونية"""
    id: str
    law_id: str
    number: str
    title: str
    content: str
    keywords: List[str] = field(default_factory=list)
    related_articles: List[str] = field(default_factory=list)
    risk_weight: float = 1.0
    embedding: Optional[np.ndarray] = None


@dataclass
class ContractClause:
    """بند عقد"""
    id: str
    contract_id: str
    number: int
    title: str
    content: str
    clause_type: str
    keywords: List[str] = field(default_factory=list)
    entities: Dict = field(default_factory=dict)
    embedding: Optional[np.ndarray] = None
    # للتدريب
    label: Optional[str] = None  # compliance label
    risk_label: Optional[str] = None  # risk level label


@dataclass
class AnalysisResult:
    """نتيجة تحليل"""
    status: ComplianceStatus
    risk_score: float
    reasoning: str
    issues: List[str] = field(default_factory=list)
    recommendations: List[str] = field(default_factory=list)
    confidence: float = 0.7


@dataclass
class LawScore:
    """نتيجة قانون"""
    law_id: str
    compliance_pct: float
    risk_score: float
    risk_level: RiskLevel
    total_checked: int
    compliant: int
    partial: int
    non_compliant: int
    reasoning: str
    findings: List[str] = field(default_factory=list)


@dataclass
class FullReport:
    """التقرير الكامل"""
    contract_id: str
    contract_name: str
    timestamp: str
    total_clauses: int
    laws_checked: List[str]
    law_scores: Dict[str, LawScore]
    overall_compliance: float
    overall_risk: float
    overall_level: RiskLevel
    summary: str
    risks: List[str] = field(default_factory=list)
    recommendations: List[str] = field(default_factory=list)
    learned_keywords: List[str] = field(default_factory=list)


@dataclass
class TrainingData:
    """بيانات التدريب"""
    X_train: np.ndarray
    X_val: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    label_encoder: LabelEncoder
    scaler: StandardScaler

print("✅ تم تعريف هياكل البيانات")

## 1.4 Knowledge Graph للقوانين

In [ ]:
class LawKnowledgeGraph:
    """Knowledge Graph للقوانين"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.embedder = SentenceTransformer(model_name)
        self.graph = nx.DiGraph()
        self.articles: Dict[str, LawArticle] = {}
        self.by_law: Dict[str, List[str]] = {law: [] for law in LAWS}
        self.index = None
        self.id_map: List[str] = []
        
        for law_id, info in LAWS.items():
            self.graph.add_node(law_id, node_type="law", **info)
    
    def add_article(self, article: LawArticle):
        """إضافة مادة"""
        text = f"{article.title} {article.content}"
        article.embedding = self.embedder.encode(text)
        
        self.articles[article.id] = article
        if article.law_id in self.by_law:
            self.by_law[article.law_id].append(article.id)
        
        self.graph.add_node(article.id, node_type="article", title=article.title)
        self.graph.add_edge(article.law_id, article.id, relation="contains")
        
        self.index = None
    
    def build_index(self):
        """بناء فهرس FAISS"""
        if not self.articles:
            return
        
        embeddings = []
        self.id_map = []
        
        for aid, art in self.articles.items():
            if art.embedding is not None:
                embeddings.append(art.embedding)
                self.id_map.append(aid)
        
        if embeddings:
            arr = np.array(embeddings).astype('float32')
            faiss.normalize_L2(arr)
            self.index = faiss.IndexFlatIP(arr.shape[1])
            self.index.add(arr)
    
    def search(self, query: str, top_k: int = 5, law_filter: List[str] = None) -> List[Tuple[LawArticle, float]]:
        """بحث"""
        if self.index is None:
            self.build_index()
        if self.index is None or not self.id_map:
            return []
        
        q = self.embedder.encode([query]).astype('float32')
        faiss.normalize_L2(q)
        
        k = min(top_k * 3, len(self.id_map))
        scores, indices = self.index.search(q, k)
        
        results = []
        for idx, score in zip(indices[0], scores[0]):
            if idx < 0:
                continue
            article = self.articles[self.id_map[idx]]
            if law_filter and article.law_id not in law_filter:
                continue
            results.append((article, float(score)))
            if len(results) >= top_k:
                break
        return results
    
    def get_all_embeddings(self) -> Tuple[np.ndarray, List[str]]:
        """استرجاع كل التضمينات للتدريب"""
        embeddings = []
        ids = []
        for aid, art in self.articles.items():
            if art.embedding is not None:
                embeddings.append(art.embedding)
                ids.append(aid)
        return np.array(embeddings) if embeddings else np.array([]), ids

print("✅ LawKnowledgeGraph")

## 1.5 Knowledge Graph للعقود

In [ ]:
class ContractKnowledgeGraph:
    """Knowledge Graph للعقود"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.embedder = SentenceTransformer(model_name)
        self.graph = nx.DiGraph()
        self.contracts: Dict[str, Dict] = {}
        self.clauses: Dict[str, ContractClause] = {}
        self.learned_patterns: Dict[str, List[str]] = {}
    
    def add_contract(self, contract_id: str, name: str, metadata: Dict = None):
        self.contracts[contract_id] = {"name": name, **(metadata or {})}
        self.graph.add_node(contract_id, node_type="contract", name=name)
    
    def add_clause(self, clause: ContractClause):
        text = f"{clause.title} {clause.content}"
        clause.embedding = self.embedder.encode(text)
        
        self.clauses[clause.id] = clause
        self.graph.add_node(clause.id, node_type="clause", title=clause.title)
        self.graph.add_edge(clause.contract_id, clause.id, relation="contains")
        
        # تعلم
        ctype = clause.clause_type
        if ctype not in self.learned_patterns:
            self.learned_patterns[ctype] = []
        for kw in clause.keywords:
            if kw not in self.learned_patterns[ctype]:
                self.learned_patterns[ctype].append(kw)
    
    def get_clauses(self, contract_id: str) -> List[ContractClause]:
        return [c for c in self.clauses.values() if c.contract_id == contract_id]
    
    def get_all_clauses_with_labels(self) -> List[ContractClause]:
        """استرجاع البنود التي لها labels للتدريب"""
        return [c for c in self.clauses.values() if c.label is not None]
    
    def get_all_embeddings(self) -> Tuple[np.ndarray, List[str], List[str]]:
        """استرجاع التضمينات مع الـ labels"""
        embeddings = []
        ids = []
        labels = []
        for cid, clause in self.clauses.items():
            if clause.embedding is not None and clause.label is not None:
                embeddings.append(clause.embedding)
                ids.append(cid)
                labels.append(clause.label)
        return np.array(embeddings) if embeddings else np.array([]), ids, labels

print("✅ ContractKnowledgeGraph")

---

# 🔍 Phase 2: التحليل المنطقي

---

## 2.1 محلل العقود (Parser)

In [ ]:
class ContractParser:
    """محلل العقود - استخراج البنود"""
    
    CLAUSE_PATTERNS = [
        r'(?:المادة|البند|الفقرة)\s*(?:رقم)?\s*[:\-]?\s*(\d+)',
        r'(?:Article|Clause|Section)\s*[:\-]?\s*(\d+)',
        r'^(\d+)[.\-\)]\s+',
    ]
    
    CLAUSE_TYPES = {
        'privacy': ['خصوصية', 'بيانات', 'سرية', 'موافقة', 'شخصية'],
        'payment': ['دفع', 'سداد', 'مقابل', 'مالي', 'ريال'],
        'liability': ['مسؤولية', 'ضمان', 'تعويض', 'إخلاء'],
        'termination': ['إنهاء', 'فسخ', 'انتهاء', 'إلغاء'],
        'security': ['أمن', 'حماية', 'اختراق', 'سيبراني'],
        'dispute': ['نزاع', 'تحكيم', 'محكمة', 'خلاف'],
        'compliance': ['امتثال', 'التزام', 'قانون', 'لائحة'],
    }
    
    def parse_text(self, text: str) -> List[Dict]:
        """استخراج البنود من نص"""
        clauses = []
        lines = text.split('\n')
        current = None
        num = 0
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            is_header = any(re.match(p, line) for p in self.CLAUSE_PATTERNS)
            is_header = is_header or (len(line) < 80 and ':' in line)
            
            if is_header:
                if current and current['content'].strip():
                    clauses.append(current)
                num += 1
                current = {
                    'number': num,
                    'title': line[:100],
                    'content': '',
                    'type': self._detect_type(line)
                }
            elif current:
                current['content'] += ' ' + line
            else:
                num += 1
                current = {'number': num, 'title': f'بند {num}', 'content': line, 'type': 'general'}
        
        if current and current['content'].strip():
            clauses.append(current)
        return clauses
    
    def _detect_type(self, text: str) -> str:
        text_lower = text.lower()
        for ctype, keywords in self.CLAUSE_TYPES.items():
            if any(kw in text_lower for kw in keywords):
                return ctype
        return 'general'
    
    def extract_keywords(self, text: str) -> List[str]:
        words = re.findall(r'[\w\u0600-\u06FF]{3,}', text)
        stopwords = {'من', 'إلى', 'على', 'في', 'أن', 'هذا', 'التي', 'الذي'}
        return list(set(w for w in words if w.lower() not in stopwords))[:15]
    
    def parse_pdf(self, path: str) -> Tuple[str, List[Dict]]:
        text = ""
        try:
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    text += (page.extract_text() or "") + "\n"
        except:
            with open(path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                for page in reader.pages:
                    text += page.extract_text() + "\n"
        return text, self.parse_text(text)
    
    def parse_pdf_folder(self, folder_path: str) -> Dict[str, Tuple[str, List[Dict]]]:
        results = {}
        folder = Path(folder_path)
        for pdf_path in list(folder.glob("*.pdf")) + list(folder.glob("*.PDF")):
            try:
                text, clauses = self.parse_pdf(str(pdf_path))
                results[pdf_path.name] = (text, clauses)
            except Exception as e:
                results[pdf_path.name] = (f"Error: {e}", [])
        return results
    
    def parse_docx(self, path: str) -> Tuple[str, List[Dict]]:
        doc = DocxDocument(path)
        text = "\n".join([p.text for p in doc.paragraphs])
        return text, self.parse_text(text)

print("✅ ContractParser")

## 2.2 محرك التحليل المنطقي (Reasoning Engine)

In [ ]:
class ReasoningEngine:
    """محرك التحليل المنطقي"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.embedder = SentenceTransformer(model_name)
    
    def analyze(self, clause: ContractClause, articles: List[LawArticle], law_id: str) -> AnalysisResult:
        """تحليل بند ضد مواد قانونية"""
        law_name = LAWS.get(law_id, {}).get('ar', law_id)
        
        if not articles:
            return AnalysisResult(
                status=ComplianceStatus.NOT_APPLICABLE,
                risk_score=25.0,
                reasoning=f"لا توجد مواد ذات صلة من {law_name}",
                confidence=0.5
            )
        
        # تحليل الكلمات
        clause_text = f"{clause.title} {clause.content}".lower()
        clause_words = set(re.findall(r'[\w\u0600-\u06FF]{3,}', clause_text))
        
        total_kw, matched_kw = 0, 0
        issues, recommendations = [], []
        
        for art in articles:
            art_kw = set(art.keywords)
            total_kw += len(art.keywords)
            matched_kw += len(clause_words & art_kw)
            
            missing = [k for k in art.keywords if k.lower() not in clause_text]
            if missing:
                issues.append(f"المادة {art.number}: مصطلحات مفقودة ({', '.join(missing[:2])})")
                recommendations.append(f"إضافة بنود عن: {', '.join(missing[:2])}")
        
        # التشابه الدلالي
        clause_emb = self.embedder.encode(f"{clause.title} {clause.content}")
        similarities = []
        for art in articles:
            if art.embedding is not None:
                sim = np.dot(clause_emb, art.embedding) / (np.linalg.norm(clause_emb) * np.linalg.norm(art.embedding) + 1e-8)
                similarities.append(float(sim))
        
        avg_sim = np.mean(similarities) if similarities else 0.3
        kw_ratio = matched_kw / max(total_kw, 1)
        combined = 0.4 * kw_ratio + 0.6 * avg_sim
        
        # تحديد الحالة
        if combined > 0.55:
            status = ComplianceStatus.COMPLIANT
            risk = 15 + (1 - combined) * 20
        elif combined > 0.35:
            status = ComplianceStatus.PARTIAL
            risk = 40 + (1 - combined) * 25
        else:
            status = ComplianceStatus.NON_COMPLIANT
            risk = 65 + (1 - combined) * 30
        
        reasoning = f"تحليل ضد {law_name}: تشابه {avg_sim*100:.0f}%، مصطلحات {kw_ratio*100:.0f}%، مجمع {combined*100:.0f}%"
        
        return AnalysisResult(
            status=status,
            risk_score=min(100, max(0, risk)),
            reasoning=reasoning,
            issues=issues[:5],
            recommendations=list(set(recommendations))[:5],
            confidence=0.6 + 0.3 * combined
        )
    
    def generate_summary(self, law_scores: Dict[str, LawScore], overall_risk: float, name: str) -> str:
        if overall_risk < 30:
            status, action = "العقد جيد", "يمكن المتابعة"
        elif overall_risk < 50:
            status, action = "يحتاج تحسينات", "مراجعة البنود المحددة"
        elif overall_risk < 75:
            status, action = "مخاطر ملحوظة", "استشارة قانونية"
        else:
            status, action = "مخاطر عالية", "تعديلات جوهرية مطلوبة"
        
        return f"العقد: {name}\nالحالة: {status}\nالمخاطر: {overall_risk:.0f}%\nالتوصية: {action}"

print("✅ ReasoningEngine")

---

# 📁 Phase 3: تحميل PDF + واجهة المستخدم

---

## 3.1 محمّل القوانين من PDF

In [ ]:
class LawLoader:
    """تحميل القوانين من PDF (ملف أو مجلد)"""
    
    def __init__(self, parser: ContractParser = None):
        self.parser = parser or ContractParser()
    
    def load_single_pdf(self, path: str, law_id: str) -> List[LawArticle]:
        """تحميل من ملف واحد"""
        text, clauses = self.parser.parse_pdf(path)
        articles = []
        for clause in clauses:
            articles.append(LawArticle(
                id=f"{law_id}_ART_{clause['number']}",
                law_id=law_id,
                number=str(clause['number']),
                title=clause['title'],
                content=clause['content'],
                keywords=self.parser.extract_keywords(clause['content'])
            ))
        return articles
    
    def load_folder(self, folder_path: str) -> Dict[str, List[LawArticle]]:
        """تحميل من مجلد"""
        results = {}
        folder = Path(folder_path)
        
        for pdf_path in list(folder.glob("*.pdf")) + list(folder.glob("*.PDF")):
            law_id = self._guess_law_id(pdf_path.stem)
            articles = self.load_single_pdf(str(pdf_path), law_id)
            if law_id not in results:
                results[law_id] = []
            results[law_id].extend(articles)
        return results
    
    def _guess_law_id(self, filename: str) -> str:
        filename_lower = filename.lower()
        patterns = {
            'PDPL': ['pdpl', 'data protection', 'حماية البيانات'],
            'ECC': ['ecc', 'e-commerce', 'تجارة إلكترونية'],
            'LABOR': ['labor', 'labour', 'عمل'],
            'CYBER': ['cyber', 'جرائم معلوماتية'],
            'COMMERCIAL': ['commercial', 'companies', 'شركات'],
            'ANTI_FRAUD': ['fraud', 'احتيال'],
        }
        for law_id, keywords in patterns.items():
            if any(kw in filename_lower for kw in keywords):
                return law_id
        return 'UNKNOWN'
    
    def load_to_kg(self, law_kg, source: str, law_id: str = None) -> int:
        """تحميل مباشر للـ KG"""
        path = Path(source)
        count = 0
        
        if path.is_file():
            if not law_id:
                law_id = self._guess_law_id(path.stem)
            for art in self.load_single_pdf(str(path), law_id):
                law_kg.add_article(art)
                count += 1
        elif path.is_dir():
            for law_id, articles in self.load_folder(str(path)).items():
                for art in articles:
                    law_kg.add_article(art)
                    count += 1
        
        law_kg.build_index()
        return count

print("✅ LawLoader")

## 3.2 المحلل الرئيسي

In [ ]:
class ContractAnalyzer:
    """المحلل الرئيسي"""
    
    def __init__(self):
        self.law_kg = LawKnowledgeGraph()
        self.contract_kg = ContractKnowledgeGraph()
        self.parser = ContractParser()
        self.loader = LawLoader(self.parser)
        self.engine = ReasoningEngine()
    
    def load_laws_from_pdf(self, source: str, law_id: str = None) -> int:
        return self.loader.load_to_kg(self.law_kg, source, law_id)
    
    def add_law_article(self, article: LawArticle):
        self.law_kg.add_article(article)
    
    def build_law_index(self):
        self.law_kg.build_index()
    
    def analyze(self, contract_text: str, contract_name: str, laws: List[str]) -> FullReport:
        """تحليل عقد"""
        contract_id = f"C_{hashlib.md5(contract_text[:50].encode()).hexdigest()[:6]}"
        self.contract_kg.add_contract(contract_id, contract_name)
        
        # استخراج البنود
        parsed = self.parser.parse_text(contract_text)
        clauses = []
        for p in parsed:
            clause = ContractClause(
                id=f"{contract_id}_CL{p['number']}",
                contract_id=contract_id,
                number=p['number'],
                title=p['title'],
                content=p['content'],
                clause_type=p['type'],
                keywords=self.parser.extract_keywords(p['content'])
            )
            self.contract_kg.add_clause(clause)
            clauses.append(clause)
        
        # تحليل
        law_results: Dict[str, List[AnalysisResult]] = {law: [] for law in laws}
        
        for clause in clauses:
            for law_id in laws:
                relevant = self.law_kg.search(clause.content, top_k=3, law_filter=[law_id])
                articles = [art for art, _ in relevant]
                result = self.engine.analyze(clause, articles, law_id)
                law_results[law_id].append(result)
        
        # حساب النتائج
        law_scores: Dict[str, LawScore] = {}
        for law_id in laws:
            results = law_results[law_id]
            if not results:
                continue
            
            compliant = sum(1 for r in results if r.status == ComplianceStatus.COMPLIANT)
            partial = sum(1 for r in results if r.status == ComplianceStatus.PARTIAL)
            non_comp = sum(1 for r in results if r.status == ComplianceStatus.NON_COMPLIANT)
            total = len(results)
            
            compliance_pct = ((compliant + partial * 0.5) / total) * 100 if total else 0
            avg_risk = np.mean([r.risk_score for r in results])
            
            level = RiskLevel.LOW if avg_risk < 30 else RiskLevel.MEDIUM if avg_risk < 50 else RiskLevel.HIGH if avg_risk < 75 else RiskLevel.CRITICAL
            
            findings = [i for r in results for i in r.issues]
            
            law_scores[law_id] = LawScore(
                law_id=law_id, compliance_pct=compliance_pct, risk_score=avg_risk,
                risk_level=level, total_checked=total, compliant=compliant,
                partial=partial, non_compliant=non_comp,
                reasoning=f"فحص {total} بند: {compliant}✅ {partial}⚠️ {non_comp}❌",
                findings=list(set(findings))[:5]
            )
        
        # النتيجة الإجمالية
        if law_scores:
            total_weight = sum(LAWS[lid]['weight'] for lid in law_scores)
            overall_compliance = sum(law_scores[lid].compliance_pct * LAWS[lid]['weight'] for lid in law_scores) / total_weight
            overall_risk = sum(law_scores[lid].risk_score * LAWS[lid]['weight'] for lid in law_scores) / total_weight
        else:
            overall_compliance, overall_risk = 50, 50
        
        overall_level = RiskLevel.LOW if overall_risk < 30 else RiskLevel.MEDIUM if overall_risk < 50 else RiskLevel.HIGH if overall_risk < 75 else RiskLevel.CRITICAL
        
        summary = self.engine.generate_summary(law_scores, overall_risk, contract_name)
        
        all_risks = [i for r_list in law_results.values() for r in r_list for i in r.issues]
        all_recs = [r for r_list in law_results.values() for res in r_list for r in res.recommendations]
        
        return FullReport(
            contract_id=contract_id, contract_name=contract_name,
            timestamp=datetime.now().isoformat(), total_clauses=len(clauses),
            laws_checked=laws, law_scores=law_scores,
            overall_compliance=overall_compliance, overall_risk=overall_risk,
            overall_level=overall_level, summary=summary,
            risks=list(set(all_risks))[:10], recommendations=list(set(all_recs))[:10],
            learned_keywords=list(set(kw for kws in self.contract_kg.learned_patterns.values() for kw in kws))[:20]
        )

print("✅ ContractAnalyzer")

## 3.3 دوال العرض

In [ ]:
def show_dashboard(report: FullReport):
    """لوحة النتائج"""
    html = f"""
    <div style="font-family: Arial; direction: rtl; padding: 20px; background: linear-gradient(135deg, #667eea, #764ba2); border-radius: 15px; color: white;">
        <h1 style="text-align: center;">📊 تقرير تحليل العقد</h1>
        <p><b>العقد:</b> {report.contract_name} | <b>البنود:</b> {report.total_clauses}</p>
        
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center; margin: 15px 0;">
            <div style="font-size: 48px; font-weight: bold; color: {report.overall_level.color};">
                {report.overall_level.emoji} {report.overall_risk:.0f}%
            </div>
            <p>نسبة المخاطر | التوافق: {report.overall_compliance:.0f}%</p>
        </div>
        
        <h3>⚖️ نتائج القوانين</h3>
        <div style="display: flex; flex-wrap: wrap; gap: 10px;">
    """
    
    for lid, s in report.law_scores.items():
        html += f"""
        <div style="background: rgba(255,255,255,0.15); padding: 12px; border-radius: 8px; flex: 1 1 250px; border-right: 4px solid {s.risk_level.color};">
            <h4 style="color: {LAWS[lid]['color']}; margin: 0;">{LAWS[lid]['ar']}</h4>
            <p style="font-size: 24px; margin: 5px 0;">المخاطر: <b style="color: {s.risk_level.color};">{s.risk_score:.0f}%</b></p>
            <p>التوافق: {s.compliance_pct:.0f}% | ✅{s.compliant} ⚠️{s.partial} ❌{s.non_compliant}</p>
        </div>
        """
    
    html += "</div></div>"
    display(HTML(html))


def show_chart(report: FullReport):
    """رسم بياني"""
    if not report.law_scores:
        return
    
    laws = [LAWS[lid]['en'] for lid in report.law_scores]
    compliance = [report.law_scores[lid].compliance_pct for lid in report.law_scores]
    risk = [report.law_scores[lid].risk_score for lid in report.law_scores]
    
    fig = go.Figure()
    fig.add_trace(go.Bar(name='Compliance', x=laws, y=compliance, marker_color='green'))
    fig.add_trace(go.Bar(name='Risk', x=laws, y=risk, marker_color='red'))
    fig.update_layout(barmode='group', height=400, template='plotly_dark', title='Per-Law Results')
    fig.show()


def save_pdf_report(report: FullReport, path: str = "report.pdf") -> str:
    """حفظ PDF"""
    doc = SimpleDocTemplate(path, pagesize=A4)
    styles = getSampleStyleSheet()
    story = [Paragraph("Contract Analysis Report", styles['Heading1']), Spacer(1, 20)]
    
    info = [["Contract:", report.contract_name], ["Risk:", f"{report.overall_risk:.0f}%"], ["Compliance:", f"{report.overall_compliance:.0f}%"]]
    story.append(Table(info, colWidths=[100, 300]))
    story.append(Spacer(1, 20))
    
    law_data = [["Law", "Compliance", "Risk", "Level"]]
    for lid, s in report.law_scores.items():
        law_data.append([LAWS[lid]['en'], f"{s.compliance_pct:.0f}%", f"{s.risk_score:.0f}%", s.risk_level.value])
    
    t = Table(law_data)
    t.setStyle(TableStyle([('BACKGROUND', (0,0), (-1,0), colors.grey), ('GRID', (0,0), (-1,-1), 0.5, colors.black)]))
    story.append(t)
    
    doc.build(story)
    return path

print("✅ دوال العرض")

---

# 🧠 Phase 4: تدريب النماذج (Train/Val/Test)

---

## 4.1 مدير بيانات التدريب

يقوم بتقسيم البيانات إلى **Train / Validation / Test** بشكل صحيح.

In [ ]:
class TrainingDataManager:
    """
    مدير بيانات التدريب
    
    يقسم البيانات إلى:
    - Train: 70% للتدريب
    - Validation: 15% للتحقق أثناء التدريب
    - Test: 15% للاختبار النهائي
    """
    
    def __init__(self, test_size: float = 0.15, val_size: float = 0.15, random_state: int = 42):
        """
        Args:
            test_size: نسبة بيانات الاختبار
            val_size: نسبة بيانات التحقق
            random_state: seed للتكرار
        """
        self.test_size = test_size
        self.val_size = val_size
        self.random_state = random_state
        self.label_encoder = LabelEncoder()
        self.scaler = StandardScaler()
    
    def prepare_data(
        self,
        X: np.ndarray,
        y: List[str],
        scale: bool = True
    ) -> TrainingData:
        """
        تجهيز البيانات وتقسيمها
        
        Args:
            X: المميزات (embeddings)
            y: التصنيفات (labels)
            scale: هل يتم تطبيع البيانات
        
        Returns:
            TrainingData مع التقسيم الكامل
        """
        # تحويل التصنيفات لأرقام
        y_encoded = self.label_encoder.fit_transform(y)
        
        # التقسيم الأول: train+val vs test
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y_encoded,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y_encoded if len(set(y_encoded)) > 1 else None
        )
        
        # التقسيم الثاني: train vs val
        # حساب نسبة val من المتبقي
        val_ratio = self.val_size / (1 - self.test_size)
        
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp,
            test_size=val_ratio,
            random_state=self.random_state,
            stratify=y_temp if len(set(y_temp)) > 1 else None
        )
        
        # التطبيع (fit على train فقط)
        if scale:
            X_train = self.scaler.fit_transform(X_train)
            X_val = self.scaler.transform(X_val)
            X_test = self.scaler.transform(X_test)
        
        return TrainingData(
            X_train=X_train, X_val=X_val, X_test=X_test,
            y_train=y_train, y_val=y_val, y_test=y_test,
            label_encoder=self.label_encoder,
            scaler=self.scaler
        )
    
    def get_split_info(self, data: TrainingData) -> Dict:
        """معلومات التقسيم"""
        total = len(data.y_train) + len(data.y_val) + len(data.y_test)
        return {
            "total": total,
            "train": len(data.y_train),
            "train_pct": len(data.y_train) / total * 100,
            "val": len(data.y_val),
            "val_pct": len(data.y_val) / total * 100,
            "test": len(data.y_test),
            "test_pct": len(data.y_test) / total * 100,
            "classes": list(self.label_encoder.classes_)
        }

print("✅ TrainingDataManager")

## 4.2 مدرب النماذج

In [ ]:
class ModelTrainer:
    """
    مدرب النماذج
    
    يدعم عدة خوارزميات:
    - Logistic Regression
    - Random Forest
    - Gradient Boosting
    - SVM
    """
    
    MODELS = {
        "logistic": lambda: LogisticRegression(max_iter=1000, random_state=42),
        "random_forest": lambda: RandomForestClassifier(n_estimators=100, random_state=42),
        "gradient_boosting": lambda: GradientBoostingClassifier(n_estimators=100, random_state=42),
        "svm": lambda: SVC(kernel='rbf', probability=True, random_state=42)
    }
    
    def __init__(self):
        self.trained_models: Dict[str, any] = {}
        self.results: Dict[str, Dict] = {}
    
    def train(
        self,
        data: TrainingData,
        model_name: str = "random_forest"
    ) -> Dict:
        """
        تدريب نموذج
        
        Args:
            data: بيانات التدريب
            model_name: اسم النموذج
        
        Returns:
            نتائج التدريب والتقييم
        """
        if model_name not in self.MODELS:
            raise ValueError(f"النموذج غير مدعوم: {model_name}")
        
        # إنشاء النموذج
        model = self.MODELS[model_name]()
        
        # التدريب
        model.fit(data.X_train, data.y_train)
        
        # التقييم على الثلاث مجموعات
        results = {
            "model_name": model_name,
            "train": self._evaluate(model, data.X_train, data.y_train, "Train"),
            "val": self._evaluate(model, data.X_val, data.y_val, "Validation"),
            "test": self._evaluate(model, data.X_test, data.y_test, "Test"),
        }
        
        # حفظ
        self.trained_models[model_name] = model
        self.results[model_name] = results
        
        return results
    
    def _evaluate(self, model, X, y, split_name: str) -> Dict:
        """تقييم النموذج"""
        y_pred = model.predict(X)
        
        return {
            "split": split_name,
            "accuracy": accuracy_score(y, y_pred),
            "precision": precision_score(y, y_pred, average='weighted', zero_division=0),
            "recall": recall_score(y, y_pred, average='weighted', zero_division=0),
            "f1": f1_score(y, y_pred, average='weighted', zero_division=0),
            "predictions": y_pred,
            "confusion_matrix": confusion_matrix(y, y_pred)
        }
    
    def train_all(self, data: TrainingData) -> Dict[str, Dict]:
        """
        تدريب جميع النماذج ومقارنتها
        """
        all_results = {}
        for model_name in self.MODELS:
            all_results[model_name] = self.train(data, model_name)
        return all_results
    
    def get_best_model(self, metric: str = "f1", split: str = "val") -> str:
        """
        اختيار أفضل نموذج بناءً على metric معين
        """
        if not self.results:
            return None
        
        best_model = None
        best_score = -1
        
        for model_name, result in self.results.items():
            score = result[split][metric]
            if score > best_score:
                best_score = score
                best_model = model_name
        
        return best_model
    
    def predict(self, X: np.ndarray, model_name: str = None) -> np.ndarray:
        """التنبؤ باستخدام نموذج محدد أو الأفضل"""
        if model_name is None:
            model_name = self.get_best_model()
        
        if model_name not in self.trained_models:
            raise ValueError(f"النموذج غير مدرب: {model_name}")
        
        return self.trained_models[model_name].predict(X)
    
    def save_model(self, model_name: str, path: str):
        """حفظ النموذج"""
        if model_name in self.trained_models:
            with open(path, 'wb') as f:
                pickle.dump(self.trained_models[model_name], f)
    
    def load_model(self, model_name: str, path: str):
        """تحميل النموذج"""
        with open(path, 'rb') as f:
            self.trained_models[model_name] = pickle.load(f)

print("✅ ModelTrainer")

## 4.3 عرض نتائج التدريب

In [ ]:
def show_training_results(results: Dict[str, Dict], data_info: Dict):
    """
    عرض نتائج التدريب بشكل مرئي
    """
    # معلومات التقسيم
    print("="*60)
    print("📊 معلومات تقسيم البيانات (Train/Val/Test)")
    print("="*60)
    print(f"   إجمالي العينات: {data_info['total']}")
    print(f"   Train: {data_info['train']} ({data_info['train_pct']:.1f}%)")
    print(f"   Validation: {data_info['val']} ({data_info['val_pct']:.1f}%)")
    print(f"   Test: {data_info['test']} ({data_info['test_pct']:.1f}%)")
    print(f"   الفئات: {data_info['classes']}")
    print()
    
    # جدول المقارنة
    print("="*60)
    print("📈 مقارنة النماذج")
    print("="*60)
    
    comparison_data = []
    for model_name, result in results.items():
        comparison_data.append({
            "Model": model_name,
            "Train Acc": f"{result['train']['accuracy']*100:.1f}%",
            "Val Acc": f"{result['val']['accuracy']*100:.1f}%",
            "Test Acc": f"{result['test']['accuracy']*100:.1f}%",
            "Val F1": f"{result['val']['f1']*100:.1f}%",
            "Test F1": f"{result['test']['f1']*100:.1f}%",
        })
    
    df = pd.DataFrame(comparison_data)
    print(df.to_string(index=False))
    print()
    
    return df


def plot_training_comparison(results: Dict[str, Dict]):
    """
    رسم بياني لمقارنة النماذج
    """
    models = list(results.keys())
    
    train_acc = [results[m]['train']['accuracy'] * 100 for m in models]
    val_acc = [results[m]['val']['accuracy'] * 100 for m in models]
    test_acc = [results[m]['test']['accuracy'] * 100 for m in models]
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(name='Train', x=models, y=train_acc, marker_color='blue'))
    fig.add_trace(go.Bar(name='Validation', x=models, y=val_acc, marker_color='orange'))
    fig.add_trace(go.Bar(name='Test', x=models, y=test_acc, marker_color='green'))
    
    fig.update_layout(
        title='مقارنة دقة النماذج على Train/Val/Test',
        barmode='group',
        yaxis_title='Accuracy %',
        height=400,
        template='plotly_dark'
    )
    
    fig.show()


def plot_confusion_matrix(cm: np.ndarray, classes: List[str], title: str = "Confusion Matrix"):
    """
    رسم مصفوفة الارتباك
    """
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

print("✅ دوال عرض التدريب")

## 4.4 نظام التدريب المتكامل

In [ ]:
class MLContractAnalyzer:
    """
    محلل العقود المعزز بالتعلم الآلي
    
    يجمع بين:
    - التحليل القائم على القواعد (Rule-based)
    - نماذج ML المدربة
    """
    
    def __init__(self):
        self.analyzer = ContractAnalyzer()
        self.data_manager = TrainingDataManager()
        self.trainer = ModelTrainer()
        self.is_trained = False
    
    def generate_training_data(
        self,
        contracts: List[Tuple[str, str, str]],  # (text, name, label)
        laws: List[str]
    ) -> Tuple[np.ndarray, List[str]]:
        """
        توليد بيانات التدريب من عقود
        
        Args:
            contracts: قائمة (نص، اسم، تصنيف)
            laws: القوانين للفحص
        
        Returns:
            (embeddings, labels)
        """
        embeddings = []
        labels = []
        
        for text, name, label in contracts:
            # تحليل العقد
            report = self.analyzer.analyze(text, name, laws)
            
            # استخراج التضمينات من البنود
            clauses = self.analyzer.contract_kg.get_clauses(report.contract_id)
            
            for clause in clauses:
                if clause.embedding is not None:
                    embeddings.append(clause.embedding)
                    labels.append(label)
                    # تعيين التصنيف للبند
                    clause.label = label
        
        return np.array(embeddings), labels
    
    def train(
        self,
        X: np.ndarray,
        y: List[str],
        model_name: str = "random_forest"
    ) -> Dict:
        """
        تدريب النموذج مع تقسيم Train/Val/Test
        """
        # تجهيز البيانات
        data = self.data_manager.prepare_data(X, y)
        split_info = self.data_manager.get_split_info(data)
        
        # التدريب
        results = self.trainer.train(data, model_name)
        results['split_info'] = split_info
        
        self.is_trained = True
        return results
    
    def train_all_models(
        self,
        X: np.ndarray,
        y: List[str]
    ) -> Dict[str, Dict]:
        """
        تدريب جميع النماذج ومقارنتها
        """
        data = self.data_manager.prepare_data(X, y)
        split_info = self.data_manager.get_split_info(data)
        
        results = self.trainer.train_all(data)
        
        # إضافة معلومات التقسيم
        results['_split_info'] = split_info
        results['_data'] = data
        
        self.is_trained = True
        return results
    
    def analyze_with_ml(
        self,
        contract_text: str,
        contract_name: str,
        laws: List[str]
    ) -> Tuple[FullReport, Optional[str]]:
        """
        تحليل مع التنبؤ ML
        
        Returns:
            (التقرير, التنبؤ ML)
        """
        # التحليل التقليدي
        report = self.analyzer.analyze(contract_text, contract_name, laws)
        
        # التنبؤ ML إذا كان النموذج مدرباً
        ml_prediction = None
        if self.is_trained:
            clauses = self.analyzer.contract_kg.get_clauses(report.contract_id)
            embeddings = [c.embedding for c in clauses if c.embedding is not None]
            
            if embeddings:
                X = self.data_manager.scaler.transform(np.array(embeddings))
                predictions = self.trainer.predict(X)
                # الأغلبية
                from collections import Counter
                ml_prediction = Counter(predictions).most_common(1)[0][0]
                ml_prediction = self.data_manager.label_encoder.inverse_transform([ml_prediction])[0]
        
        return report, ml_prediction

print("✅ MLContractAnalyzer")

---

# 🧪 الاختبار

---

## اختبار 1: تحميل القوانين

In [ ]:
# إنشاء المحلل
ml_analyzer = MLContractAnalyzer()

# تحميل قوانين عينة
sample_laws = {
    "PDPL": [
        ("5", "الموافقة", "يجب الحصول على موافقة صريحة قبل معالجة البيانات الشخصية", ["موافقة", "بيانات", "معالجة"]),
        ("10", "الحقوق", "لصاحب البيانات حق الوصول والتصحيح والحذف", ["حقوق", "وصول", "حذف"]),
        ("15", "النقل", "يُحظر نقل البيانات خارج المملكة بدون ضمانات", ["نقل", "خارج", "ضمانات"]),
    ],
    "CYBER": [
        ("3", "الاختراق", "يُعاقب من يدخل نظاماً بدون إذن", ["اختراق", "دخول", "نظام"]),
        ("5", "التنصت", "يُعاقب من يتنصت على البيانات", ["تنصت", "بيانات", "شبكة"]),
    ],
    "ECC": [
        ("3", "الإفصاح", "يجب توفير معلومات واضحة عن مقدم الخدمة", ["إفصاح", "معلومات", "هوية"]),
        ("7", "المستهلك", "يحق للمستهلك الإلغاء خلال 7 أيام", ["مستهلك", "إلغاء", "استرجاع"]),
    ],
    "LABOR": [
        ("74", "العقد", "يجب أن يكون عقد العمل مكتوباً ويتضمن الأجر", ["عقد", "أجر", "عمل"]),
        ("84", "المكافأة", "يستحق العامل مكافأة نهاية الخدمة", ["مكافأة", "خدمة"]),
    ],
}

for law_id, articles in sample_laws.items():
    for num, title, content, keywords in articles:
        ml_analyzer.analyzer.add_law_article(LawArticle(
            id=f"{law_id}_{num}", law_id=law_id, number=num,
            title=title, content=content, keywords=keywords
        ))

ml_analyzer.analyzer.build_law_index()
print(f"✅ تم تحميل {len(ml_analyzer.analyzer.law_kg.articles)} مادة قانونية")

## اختبار 2: تحليل عقد

In [ ]:
test_contract = """
عقد خدمات تقنية المعلومات

المادة الأولى: التعريفات
يُقصد بالمصطلحات التالية المعاني المبينة.

المادة الثانية: حماية البيانات
يلتزم الطرف الثاني بالحفاظ على سرية البيانات الشخصية والحصول على موافقة العملاء قبل أي معالجة.

المادة الثالثة: الأمن السيبراني
يلتزم الطرف الثاني باتخاذ التدابير الأمنية لحماية الأنظمة من الاختراق.

المادة الرابعة: المقابل المالي
يلتزم الطرف الأول بسداد 100,000 ريال مقابل الخدمات.

المادة الخامسة: المسؤولية
يتحمل الطرف الثاني المسؤولية عن أي تسريب للبيانات.
"""

report = ml_analyzer.analyzer.analyze(
    contract_text=test_contract,
    contract_name="عقد خدمات تقنية",
    laws=["PDPL", "ECC", "LABOR", "CYBER"]
)

show_dashboard(report)
show_chart(report)

## اختبار 3: تدريب النماذج (Train/Val/Test)

In [ ]:
# توليد بيانات تدريب وهمية
# في الواقع ستستخدم عقود حقيقية مع تصنيفاتها

sample_contracts = [
    # (نص، اسم، تصنيف المخاطر)
    ("يلتزم الطرف بحماية البيانات والحصول على موافقة صريحة", "عقد 1", "low_risk"),
    ("يلتزم بالسرية التامة والأمن السيبراني والحماية", "عقد 2", "low_risk"),
    ("بنود عامة بدون ذكر حماية أو موافقة", "عقد 3", "high_risk"),
    ("لا يوجد بند للخصوصية أو البيانات", "عقد 4", "high_risk"),
    ("حماية جزئية للبيانات بدون موافقة صريحة", "عقد 5", "medium_risk"),
    ("بنود أمنية بدون تفاصيل كافية", "عقد 6", "medium_risk"),
    ("التزام كامل بجميع متطلبات الخصوصية والأمن", "عقد 7", "low_risk"),
    ("عدم وجود أي ضمانات قانونية", "عقد 8", "high_risk"),
    ("حماية متوسطة مع بعض الثغرات", "عقد 9", "medium_risk"),
    ("التزام تام بالقوانين والمعايير", "عقد 10", "low_risk"),
    ("بنود ضعيفة وغير واضحة", "عقد 11", "high_risk"),
    ("حماية جيدة مع بعض النقص", "عقد 12", "medium_risk"),
]

# توليد بيانات التدريب
X, y = ml_analyzer.generate_training_data(sample_contracts, ["PDPL", "CYBER"])
print(f"📊 بيانات التدريب: {X.shape[0]} عينة، {X.shape[1]} ميزة")

In [ ]:
# تدريب جميع النماذج
results = ml_analyzer.train_all_models(X, y)

# عرض النتائج
split_info = results.pop('_split_info')
data = results.pop('_data')

df = show_training_results(results, split_info)

In [ ]:
# رسم بياني للمقارنة
plot_training_comparison(results)

In [ ]:
# أفضل نموذج
best = ml_analyzer.trainer.get_best_model(metric='f1', split='val')
print(f"\n🏆 أفضل نموذج: {best}")
print(f"   Val F1: {results[best]['val']['f1']*100:.1f}%")
print(f"   Test F1: {results[best]['test']['f1']*100:.1f}%")

In [ ]:
# مصفوفة الارتباك للنموذج الأفضل
cm = results[best]['test']['confusion_matrix']
classes = split_info['classes']
plot_confusion_matrix(cm, classes, f"Confusion Matrix - {best} (Test Set)")

## اختبار 4: تحليل مع ML

In [ ]:
# تحليل عقد جديد مع التنبؤ ML
new_contract = """
عقد جديد للخدمات

المادة الأولى:
يلتزم المزود بحماية البيانات الشخصية والحصول على موافقة صريحة.

المادة الثانية:
يجب اتخاذ كافة الإجراءات الأمنية لحماية الأنظمة من الاختراق.
"""

report, ml_prediction = ml_analyzer.analyze_with_ml(
    new_contract, "عقد جديد", ["PDPL", "CYBER"]
)

print(f"📊 نتيجة التحليل التقليدي: {report.overall_level.arabic} ({report.overall_risk:.0f}%)")
if ml_prediction:
    print(f"🤖 تنبؤ ML: {ml_prediction}")

---

# 🖥️ واجهة Gradio

In [ ]:
import gradio as gr

def analyze_ui(contract_text, contract_name, selected_laws, contract_file, law_folder):
    """دالة التحليل للواجهة"""
    # تحميل قوانين إضافية
    if law_folder and os.path.exists(law_folder):
        try:
            ml_analyzer.analyzer.load_laws_from_pdf(law_folder)
        except:
            pass
    
    # قراءة الملف
    if contract_file:
        try:
            if contract_file.name.endswith('.pdf'):
                text, _ = ml_analyzer.analyzer.parser.parse_pdf(contract_file.name)
            elif contract_file.name.endswith('.docx'):
                text, _ = ml_analyzer.analyzer.parser.parse_docx(contract_file.name)
            else:
                with open(contract_file.name, 'r', encoding='utf-8') as f:
                    text = f.read()
            contract_text = text
        except Exception as e:
            return f"❌ خطأ: {e}", None, None
    
    if not contract_text or not contract_text.strip():
        return "❌ أدخل نص العقد أو ارفع ملف", None, None
    
    if not selected_laws:
        return "❌ اختر قانون واحد على الأقل", None, None
    
    # التحليل
    report, ml_pred = ml_analyzer.analyze_with_ml(contract_text, contract_name or "عقد", selected_laws)
    
    # النتائج
    result = f"{report.summary}\n\n"
    for lid, s in report.law_scores.items():
        result += f"{s.risk_level.emoji} {LAWS[lid]['ar']}: توافق {s.compliance_pct:.0f}%, مخاطر {s.risk_score:.0f}%\n"
    
    if ml_pred:
        result += f"\n🤖 تنبؤ ML: {ml_pred}"
    
    # الرسم
    if report.law_scores:
        laws = [LAWS[lid]['en'] for lid in report.law_scores]
        compliance = [report.law_scores[lid].compliance_pct for lid in report.law_scores]
        risk = [report.law_scores[lid].risk_score for lid in report.law_scores]
        
        fig = go.Figure()
        fig.add_trace(go.Bar(name='Compliance', x=laws, y=compliance, marker_color='green'))
        fig.add_trace(go.Bar(name='Risk', x=laws, y=risk, marker_color='red'))
        fig.update_layout(barmode='group', height=350, template='plotly_dark')
    else:
        fig = None
    
    # PDF
    pdf_path = save_pdf_report(report, "analysis_report.pdf")
    
    return result, fig, pdf_path


# بناء الواجهة
with gr.Blocks(title="📜 Contract Analyzer", theme=gr.themes.Soft(primary_hue="purple")) as demo:
    gr.Markdown("# 📜 نظام تحليل العقود (Phase 1-4)\n### مع تدريب ML وتقسيم Train/Val/Test")
    
    with gr.Row():
        with gr.Column():
            contract_name = gr.Textbox(label="اسم العقد", placeholder="عقد خدمات...")
            contract_text = gr.Textbox(label="نص العقد", lines=8)
            contract_file = gr.File(label="📄 رفع ملف", file_types=[".pdf", ".docx", ".txt"])
            selected_laws = gr.CheckboxGroup(choices=list(LAWS.keys()), value=["PDPL", "CYBER"], label="القوانين")
            law_folder = gr.Textbox(label="📁 مجلد القوانين (اختياري)")
            analyze_btn = gr.Button("🔍 تحليل", variant="primary")
        
        with gr.Column():
            result_text = gr.Textbox(label="النتائج", lines=12)
            result_chart = gr.Plot(label="الرسم البياني")
            pdf_file = gr.File(label="📥 PDF")
    
    analyze_btn.click(analyze_ui, [contract_text, contract_name, selected_laws, contract_file, law_folder], [result_text, result_chart, pdf_file])

In [ ]:
# تشغيل الواجهة
demo.launch(share=True)

---

# 🔬 Phase 4 المتقدم: Cross-Validation و Hyperparameter Tuning

---

## 4.5 Cross-Validation مع Train/Val/Test

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

class AdvancedTrainer:
    """
    مدرب متقدم مع:
    - Cross-Validation
    - Hyperparameter Tuning
    - Early Stopping simulation
    """
    
    def __init__(self, n_folds: int = 5):
        self.n_folds = n_folds
        self.best_params = {}
        self.cv_results = {}
    
    def cross_validate(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        model_name: str = "random_forest"
    ) -> Dict:
        """
        Cross-validation على بيانات التدريب فقط
        
        ملاحظة: Test set يبقى منفصلاً تماماً!
        """
        models = {
            "logistic": LogisticRegression(max_iter=1000, random_state=42),
            "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
            "gradient_boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
            "svm": SVC(kernel='rbf', random_state=42)
        }
        
        model = models[model_name]
        
        # Stratified K-Fold
        cv = StratifiedKFold(n_splits=self.n_folds, shuffle=True, random_state=42)
        
        # Cross-validation scores
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_weighted')
        
        result = {
            "model": model_name,
            "cv_scores": scores,
            "mean_cv_score": scores.mean(),
            "std_cv_score": scores.std(),
            "n_folds": self.n_folds
        }
        
        self.cv_results[model_name] = result
        return result
    
    def hyperparameter_tuning(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        model_name: str = "random_forest"
    ) -> Dict:
        """
        Grid Search لإيجاد أفضل المعاملات
        يستخدم Cross-Validation داخلياً
        """
        param_grids = {
            "logistic": {
                "C": [0.1, 1, 10],
                "penalty": ["l2"]
            },
            "random_forest": {
                "n_estimators": [50, 100, 200],
                "max_depth": [None, 10, 20],
                "min_samples_split": [2, 5]
            },
            "gradient_boosting": {
                "n_estimators": [50, 100],
                "learning_rate": [0.05, 0.1, 0.2],
                "max_depth": [3, 5]
            },
            "svm": {
                "C": [0.1, 1, 10],
                "kernel": ["rbf", "linear"]
            }
        }
        
        models = {
            "logistic": LogisticRegression(max_iter=1000, random_state=42),
            "random_forest": RandomForestClassifier(random_state=42),
            "gradient_boosting": GradientBoostingClassifier(random_state=42),
            "svm": SVC(random_state=42)
        }
        
        grid_search = GridSearchCV(
            models[model_name],
            param_grids[model_name],
            cv=3,  # 3-fold للسرعة
            scoring='f1_weighted',
            n_jobs=-1
        )
        
        grid_search.fit(X_train, y_train)
        
        self.best_params[model_name] = grid_search.best_params_
        
        return {
            "model": model_name,
            "best_params": grid_search.best_params_,
            "best_score": grid_search.best_score_,
            "best_estimator": grid_search.best_estimator_
        }

print("✅ AdvancedTrainer")

## 4.6 تقييم شامل مع التقسيم الثلاثي

In [ ]:
class ComprehensiveEvaluator:
    """
    مقيّم شامل يضمن التقسيم الصحيح:
    
    Train (70%) → للتدريب
    Val (15%)   → لاختيار النموذج والمعاملات
    Test (15%)  → للتقييم النهائي فقط (لا يُلمس أبداً أثناء التدريب!)
    """
    
    def __init__(self):
        self.history = []
    
    def full_evaluation_pipeline(
        self,
        X: np.ndarray,
        y: np.ndarray,
        test_size: float = 0.15,
        val_size: float = 0.15
    ) -> Dict:
        """
        خط أنابيب التقييم الكامل
        
        الخطوات:
        1. تقسيم البيانات إلى Train/Val/Test
        2. تدريب النماذج على Train
        3. اختيار أفضل نموذج باستخدام Val
        4. التقييم النهائي على Test
        """
        print("="*60)
        print("🔬 بدء خط أنابيب التقييم الشامل")
        print("="*60)
        
        # === الخطوة 1: التقسيم ===
        print("\n📊 الخطوة 1: تقسيم البيانات (Train/Val/Test)")
        
        # أولاً: فصل Test
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        
        # ثانياً: فصل Val من المتبقي
        val_ratio = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_ratio, random_state=42, stratify=y_temp
        )
        
        print(f"   Train: {len(y_train)} samples ({len(y_train)/len(y)*100:.1f}%)")
        print(f"   Val:   {len(y_val)} samples ({len(y_val)/len(y)*100:.1f}%)")
        print(f"   Test:  {len(y_test)} samples ({len(y_test)/len(y)*100:.1f}%)")
        
        # تطبيع
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)  # fit على Train فقط!
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        
        # === الخطوة 2: تدريب النماذج ===
        print("\n🏋️ الخطوة 2: تدريب النماذج على Train")
        
        models = {
            "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
            "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
            "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
            "SVM": SVC(kernel='rbf', probability=True, random_state=42)
        }
        
        results = {}
        for name, model in models.items():
            # تدريب
            model.fit(X_train_scaled, y_train)
            
            # تقييم على كل المجموعات
            train_pred = model.predict(X_train_scaled)
            val_pred = model.predict(X_val_scaled)
            test_pred = model.predict(X_test_scaled)
            
            results[name] = {
                "model": model,
                "train": {
                    "accuracy": accuracy_score(y_train, train_pred),
                    "f1": f1_score(y_train, train_pred, average='weighted', zero_division=0)
                },
                "val": {
                    "accuracy": accuracy_score(y_val, val_pred),
                    "f1": f1_score(y_val, val_pred, average='weighted', zero_division=0)
                },
                "test": {
                    "accuracy": accuracy_score(y_test, test_pred),
                    "f1": f1_score(y_test, test_pred, average='weighted', zero_division=0),
                    "predictions": test_pred,
                    "confusion_matrix": confusion_matrix(y_test, test_pred)
                }
            }
            print(f"   ✓ {name}: Train F1={results[name]['train']['f1']:.3f}, Val F1={results[name]['val']['f1']:.3f}")
        
        # === الخطوة 3: اختيار أفضل نموذج (بناءً على Val فقط!) ===
        print("\n🏆 الخطوة 3: اختيار أفضل نموذج (بناءً على Validation)")
        
        best_model_name = max(results, key=lambda x: results[x]['val']['f1'])
        best_val_f1 = results[best_model_name]['val']['f1']
        print(f"   أفضل نموذج: {best_model_name} (Val F1: {best_val_f1:.3f})")
        
        # === الخطوة 4: التقييم النهائي على Test ===
        print("\n📋 الخطوة 4: التقييم النهائي على Test (مرة واحدة فقط!)")
        
        best_test_results = results[best_model_name]['test']
        print(f"   Test Accuracy: {best_test_results['accuracy']:.3f}")
        print(f"   Test F1 Score: {best_test_results['f1']:.3f}")
        
        # فحص Overfitting
        train_f1 = results[best_model_name]['train']['f1']
        test_f1 = best_test_results['f1']
        gap = train_f1 - test_f1
        
        print(f"\n⚠️ فحص Overfitting:")
        print(f"   Train-Test Gap: {gap:.3f}")
        if gap > 0.1:
            print("   ⚠️ تحذير: قد يكون هناك overfitting!")
        else:
            print("   ✅ النموذج يعمم بشكل جيد")
        
        print("\n" + "="*60)
        print("✅ اكتمل التقييم")
        print("="*60)
        
        return {
            "all_results": results,
            "best_model_name": best_model_name,
            "best_model": results[best_model_name]['model'],
            "final_test_f1": best_test_results['f1'],
            "final_test_accuracy": best_test_results['accuracy'],
            "scaler": scaler,
            "split_info": {
                "train_size": len(y_train),
                "val_size": len(y_val),
                "test_size": len(y_test)
            }
        }

print("✅ ComprehensiveEvaluator")

## 4.7 عرض نتائج التقسيم الثلاثي

In [ ]:
def plot_train_val_test_comparison(results: Dict):
    """
    رسم مقارنة شاملة للثلاث مجموعات
    """
    models = list(results['all_results'].keys())
    
    train_f1 = [results['all_results'][m]['train']['f1'] * 100 for m in models]
    val_f1 = [results['all_results'][m]['val']['f1'] * 100 for m in models]
    test_f1 = [results['all_results'][m]['test']['f1'] * 100 for m in models]
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        name='Train',
        x=models, y=train_f1,
        marker_color='#3498db',
        text=[f'{v:.1f}%' for v in train_f1],
        textposition='auto'
    ))
    
    fig.add_trace(go.Bar(
        name='Validation',
        x=models, y=val_f1,
        marker_color='#f39c12',
        text=[f'{v:.1f}%' for v in val_f1],
        textposition='auto'
    ))
    
    fig.add_trace(go.Bar(
        name='Test',
        x=models, y=test_f1,
        marker_color='#2ecc71',
        text=[f'{v:.1f}%' for v in test_f1],
        textposition='auto'
    ))
    
    # إضافة خط لأفضل نموذج
    best_idx = models.index(results['best_model_name'])
    fig.add_annotation(
        x=results['best_model_name'],
        y=max(train_f1[best_idx], val_f1[best_idx], test_f1[best_idx]) + 5,
        text="🏆 الأفضل",
        showarrow=True,
        arrowhead=2
    )
    
    fig.update_layout(
        title='📊 مقارنة F1 Score: Train vs Validation vs Test',
        barmode='group',
        yaxis_title='F1 Score %',
        height=450,
        template='plotly_dark',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    
    fig.show()


def display_split_summary(results: Dict):
    """
    عرض ملخص التقسيم
    """
    split = results['split_info']
    total = split['train_size'] + split['val_size'] + split['test_size']
    
    html = f"""
    <div style="font-family: Arial; padding: 20px; background: linear-gradient(135deg, #1a1a2e, #16213e); border-radius: 15px; color: white;">
        <h2 style="text-align: center;">📊 ملخص تقسيم البيانات (Train/Val/Test)</h2>
        
        <div style="display: flex; justify-content: space-around; margin: 20px 0;">
            <div style="text-align: center; background: rgba(52, 152, 219, 0.3); padding: 20px; border-radius: 10px; flex: 1; margin: 5px;">
                <div style="font-size: 36px; font-weight: bold; color: #3498db;">🏋️</div>
                <div style="font-size: 24px; font-weight: bold;">{split['train_size']}</div>
                <div>Train ({split['train_size']/total*100:.0f}%)</div>
            </div>
            
            <div style="text-align: center; background: rgba(243, 156, 18, 0.3); padding: 20px; border-radius: 10px; flex: 1; margin: 5px;">
                <div style="font-size: 36px; font-weight: bold; color: #f39c12;">🔍</div>
                <div style="font-size: 24px; font-weight: bold;">{split['val_size']}</div>
                <div>Validation ({split['val_size']/total*100:.0f}%)</div>
            </div>
            
            <div style="text-align: center; background: rgba(46, 204, 113, 0.3); padding: 20px; border-radius: 10px; flex: 1; margin: 5px;">
                <div style="font-size: 36px; font-weight: bold; color: #2ecc71;">📋</div>
                <div style="font-size: 24px; font-weight: bold;">{split['test_size']}</div>
                <div>Test ({split['test_size']/total*100:.0f}%)</div>
            </div>
        </div>
        
        <hr style="border-color: rgba(255,255,255,0.2);">
        
        <h3>🏆 النتيجة النهائية</h3>
        <p><b>أفضل نموذج:</b> {results['best_model_name']}</p>
        <p><b>Test F1 Score:</b> <span style="color: #2ecc71; font-size: 20px;">{results['final_test_f1']*100:.1f}%</span></p>
        <p><b>Test Accuracy:</b> <span style="color: #2ecc71; font-size: 20px;">{results['final_test_accuracy']*100:.1f}%</span></p>
        
        <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 8px; margin-top: 15px;">
            <h4>📌 ملاحظات مهمة:</h4>
            <ul>
                <li>✅ Train: تُستخدم للتدريب فقط</li>
                <li>✅ Validation: لاختيار أفضل نموذج ومعاملات</li>
                <li>✅ Test: للتقييم النهائي مرة واحدة فقط!</li>
            </ul>
        </div>
    </div>
    """
    display(HTML(html))

print("✅ دوال العرض المتقدمة")

## 4.8 اختبار التقسيم الثلاثي الكامل

In [ ]:
# توليد بيانات أكثر للاختبار
np.random.seed(42)

# محاكاة بيانات
n_samples = 100
n_features = 384  # حجم embedding من all-MiniLM-L6-v2

X_synthetic = np.random.randn(n_samples, n_features)
y_synthetic = np.random.choice(['low_risk', 'medium_risk', 'high_risk'], n_samples)

# تحويل التصنيفات
le = LabelEncoder()
y_encoded = le.fit_transform(y_synthetic)

print(f"📊 بيانات التجربة:")
print(f"   العينات: {n_samples}")
print(f"   الميزات: {n_features}")
print(f"   الفئات: {list(le.classes_)}")

In [ ]:
# تشغيل خط الأنابيب الكامل
evaluator = ComprehensiveEvaluator()
results = evaluator.full_evaluation_pipeline(
    X_synthetic, 
    y_encoded,
    test_size=0.15,  # 15% للاختبار
    val_size=0.15    # 15% للتحقق
)

In [ ]:
# عرض ملخص التقسيم
display_split_summary(results)

In [ ]:
# رسم المقارنة
plot_train_val_test_comparison(results)

In [ ]:
# مصفوفة الارتباك للنموذج الأفضل
best_cm = results['all_results'][results['best_model_name']]['test']['confusion_matrix']

plt.figure(figsize=(8, 6))
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f"Confusion Matrix - {results['best_model_name']} (Test Set)")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 4.9 حفظ وتحميل النموذج

In [ ]:
def save_trained_model(results: Dict, path: str = "trained_model.pkl"):
    """
    حفظ النموذج المدرب مع المكونات اللازمة
    """
    to_save = {
        "model": results['best_model'],
        "model_name": results['best_model_name'],
        "scaler": results['scaler'],
        "test_f1": results['final_test_f1'],
        "test_accuracy": results['final_test_accuracy'],
        "timestamp": datetime.now().isoformat()
    }
    
    with open(path, 'wb') as f:
        pickle.dump(to_save, f)
    
    print(f"✅ تم حفظ النموذج: {path}")
    return path


def load_trained_model(path: str = "trained_model.pkl") -> Dict:
    """
    تحميل النموذج المحفوظ
    """
    with open(path, 'rb') as f:
        loaded = pickle.load(f)
    
    print(f"✅ تم تحميل النموذج: {loaded['model_name']}")
    print(f"   Test F1: {loaded['test_f1']*100:.1f}%")
    return loaded

# حفظ النموذج
save_trained_model(results, "best_contract_model.pkl")

---

# 📋 ملخص الاستخدام

```python
# 1. إنشاء المحلل
ml_analyzer = MLContractAnalyzer()

# 2. تحميل القوانين
ml_analyzer.analyzer.add_law_article(LawArticle(...))
# أو من PDF
ml_analyzer.analyzer.load_laws_from_pdf("laws_folder/")

# 3. توليد بيانات التدريب
X, y = ml_analyzer.generate_training_data(contracts, laws)

# 4. تدريب النماذج (Train/Val/Test)
evaluator = ComprehensiveEvaluator()
results = evaluator.full_evaluation_pipeline(X, y)

# 5. عرض النتائج
display_split_summary(results)
plot_train_val_test_comparison(results)

# 6. حفظ النموذج
save_trained_model(results)

# 7. تحليل مع ML
report, ml_pred = ml_analyzer.analyze_with_ml(text, name, laws)
```

---

## 📊 تقسيم البيانات الصحيح:

| المجموعة | النسبة | الاستخدام |
|----------|--------|------------|
| **Train** | 70% | تدريب النموذج |
| **Validation** | 15% | اختيار النموذج والمعاملات |
| **Test** | 15% | التقييم النهائي (مرة واحدة!) |

### ⚠️ قواعد مهمة:
1. **لا تلمس Test** أثناء التدريب أو اختيار المعاملات
2. **استخدم Validation** لاختيار أفضل نموذج
3. **Test للتقييم النهائي فقط** - مرة واحدة في النهاية

---